# DDS-Edge: train on Kaggle (cleaned 320x240 dataset)

Before running: **Add Input** -> your dataset with `sf_320x240` (images + manifest.csv); do NOT add the original State Farm competition. Session options -> **Accelerator: GPU (P100)**, **Internet: On**.

In [ ]:
import torch
assert torch.cuda.is_available(), 'Bat GPU: Session options -> Accelerator -> GPU'
print(torch.cuda.get_device_name(0))

In [ ]:
%%bash
cd /kaggle/working
[ -d dds-edge ] || git clone https://github.com/ColorOfDreams/dds-edge.git
cd dds-edge && git pull -q
pip install -q timm onnx onnxruntime
pip install -q -e . --no-deps
git log --oneline -1

In [ ]:
# Link the uploaded dataset to where the manifest paths expect it: data/processed/sf_320x240
import os, glob
os.chdir('/kaggle/working/dds-edge')
hits = glob.glob('/kaggle/input/**/manifest.csv', recursive=True)
if not hits:  # dataset uploaded as a .zip that Kaggle did not extract
    zips = glob.glob('/kaggle/input/**/*.zip', recursive=True)
    assert zips, 'Khong thay manifest.csv hay file .zip: da Add Input dataset sf_320x240 chua?'
    import zipfile, shutil
    print('extracting', zips[0])
    with zipfile.ZipFile(zips[0]) as z:
        for info in z.infolist():
            name = info.filename.replace(chr(92), '/')  # zips made on Windows use backslashes
            if name.endswith('/'):
                continue
            dst = os.path.join('/kaggle/working/sf_data', name)
            os.makedirs(os.path.dirname(dst), exist_ok=True)
            with z.open(info) as f, open(dst, 'wb') as g:
                shutil.copyfileobj(f, g)
    hits = glob.glob('/kaggle/working/sf_data/**/manifest.csv', recursive=True)
assert hits, 'Khong thay manifest.csv trong dataset'
src = os.path.dirname(hits[0])
print('dataset at', src, os.listdir(src))
os.makedirs('data/processed', exist_ok=True)
link = 'data/processed/sf_320x240'
if os.path.islink(link):  # stale or broken link from an earlier run
    os.unlink(link)
assert not os.path.exists(link), f'{link} is a real folder, remove it first'
os.symlink(src, link)
import pandas as pd
m = pd.read_csv(link + '/manifest.csv')
print(len(m), 'images;', m.groupby('split')['group'].nunique().to_dict(), 'drivers per split')

In [ ]:
%cd /kaggle/working/dds-edge
# smoke test (1 epoch) - skip once it works
!python -m dds_edge.dl.train --epochs 1 --manifest processed/sf_320x240/manifest.csv --name smoke

In [ ]:
%cd /kaggle/working/dds-edge
# baseline on cleaned data: MobileNetV3-Small, pretrained, aug, grouped6
!python -m dds_edge.dl.train --arch mobilenet_v3_small --grouping grouped6 --manifest processed/sf_320x240/manifest.csv --name mnv3_clean

In [ ]:
%cd /kaggle/working/dds-edge
# baselines on the clean data are DONE (runlog 3-6): mnv3_clean, resnet18_clean, effl0_clean, resnet50_clean.
# Re-run only if the session was lost (best.pt is needed for INT8). ResNet50 is a reference only: no improvement runs for it.
# !python -m dds_edge.dl.train --arch resnet18 --grouping grouped6 --manifest processed/sf_320x240/manifest.csv --name resnet18_clean
# !python -m dds_edge.dl.train --arch efficientnet_lite0 --grouping grouped6 --manifest processed/sf_320x240/manifest.csv --name effl0_clean
# optional reference, run last:
# !python -m dds_edge.dl.train --arch vgg16 --grouping grouped6 --manifest processed/sf_320x240/manifest.csv --name vgg16_clean

In [ ]:
%cd /kaggle/working/dds-edge
# improvement experiments: change ONE thing vs the baseline of the same model. Uncomment ONE line per run.
# Plan: sweep on mobilenet_v3_small, then run the BEST setting on efficientnet_lite0 and resnet18 (not resnet50).
# Order for the sweep: cw_sqrt -> lr3e-4 -> lr3e-3 -> wd1e-2 -> (cw_inv, ls0 if time).
ARCH = "mobilenet_v3_small"   # then "efficientnet_lite0" / "resnet18"
TAG = "mnv3"                  # run-name prefix, e.g. "effl0" / "resnet18"
# baseline to compare with: runs/mnv3_clean (0.7524 test macro-F1), effl0_clean (0.7860), resnet18_clean (0.7517)

# class imbalance (talk_passenger / normal are weak)
# !python -m dds_edge.dl.train --arch {ARCH} --grouping grouped6 --manifest processed/sf_320x240/manifest.csv --class_weight sqrt --name {TAG}_cw_sqrt
# !python -m dds_edge.dl.train --arch {ARCH} --grouping grouped6 --manifest processed/sf_320x240/manifest.csv --class_weight inverse --name {TAG}_cw_inv
# learning rate (baseline 1e-3)
# !python -m dds_edge.dl.train --arch {ARCH} --grouping grouped6 --manifest processed/sf_320x240/manifest.csv --lr 3e-4 --name {TAG}_lr3e-4
# !python -m dds_edge.dl.train --arch {ARCH} --grouping grouped6 --manifest processed/sf_320x240/manifest.csv --lr 3e-3 --name {TAG}_lr3e-3
# regularization (baseline weight_decay 1e-4, label_smoothing 0.1)
# !python -m dds_edge.dl.train --arch {ARCH} --grouping grouped6 --manifest processed/sf_320x240/manifest.csv --weight_decay 1e-2 --name {TAG}_wd1e-2
# !python -m dds_edge.dl.train --arch {ARCH} --grouping grouped6 --manifest processed/sf_320x240/manifest.csv --label_smoothing 0.0 --name {TAG}_ls0

In [ ]:
# package results: download runs.zip from the Output tab (/kaggle/working)
!cd /kaggle/working/dds-edge && zip -qr /kaggle/working/runs.zip runs data/processed/manifest.csv
!ls -lh /kaggle/working/runs.zip